# qang on IBM device noise models: the latest results (§145–§150)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_ibm_recent.ipynb)

A short, runnable version of RESEARCH_NOTES §145–§150. Everything runs on **local simulators of IBM devices** (the `Fake*` backends of `qiskit-ibm-runtime`, with their calibrated noise models): no account, no token, no quantum hardware.

Each section compares the read-out **without qang** (raw shots) and **with qang** (the qg filter: keep only the shots in the Hamming-weight sector the circuit conserves, optionally with readout correction). The published runs used 3–21 backends and up to 8000 shots; here 2 backends and fewer shots/circuits, so the numbers are noisier but show the same picture. The whole notebook takes about 15 minutes on a laptop CPU (most of it in §148 and §149).

| § | topic | published finding |
|---|---|---|
| 145 | coherent RZZ(ε) on the native two-qubit gate | partly filtered: it acts inside the decomposition of XX+YY |
| 146 | a PCA spectrum read from the weight-1 sector | filter + readout: L1 error 4.6× lower |
| 147 | Trotter simulation and a quantum walk on the XY chain | the filter allows deeper circuits; the walk stays ballistic |
| 148 | the XXZ chain with 1, 2 and 3 excitations | the filter's gain falls with the number of excitations |
| 149 | swap test vs inversion test | filter helps the inversion test 4–11×, the swap test only 1.4–1.8× |
| 150 | always-on ZZ crosstalk | only partly invisible to the filter |

## 1. Install

In [1]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang", "qiskit-aer", "qiskit-ibm-runtime", "scikit-learn"], check=False)
importlib.invalidate_caches()
try:
    from qang import ibm
    ok = all(hasattr(ibm, f) for f in ("add_static_zz", "xxz_trotter", "swap_test_circuit"))
except ImportError:
    ok = False
if not ok:  # the published qang predates §145-§150: install from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/Viny2030/qang.git"], check=True)
    print("installed qang from GitHub; if an import fails below, restart the runtime once")
import qang
print("qang", qang.__version__)

qang 0.10.0


## 2. Shared set-up

`circuit(seed)` and `ideal(qc)` are the ten 5-qubit weight-1 circuits of §137 (one excitation moved along a chain by random XX+YY gates; copied from `examples/ibm_runtime_qg.py`, which is not part of the pip package). `KINDS` are the four read-outs of `qang.ibm.ibm_qg`: raw, readout-corrected, filtered, filtered + readout.

In [2]:
import time
import numpy as np
import qiskit_ibm_runtime.fake_provider as fp
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import XXPlusYYGate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qang import ibm
from qang import interop as IO

N = 5
BACKENDS = ("FakeBrisbane", "FakeTorino")   # ECR and CZ native gates
KINDS = ("raw", "readout", "filtered", "filtered_readout")
FR = "filtered_readout"
T0 = time.time()


def circuit(seed):
    rng = np.random.default_rng(seed)
    qc = QuantumCircuit(N)
    qc.x(0)
    for _ in range(2):
        for i in range(N - 1):
            qc.append(XXPlusYYGate(rng.uniform(0.3, 2.8)), [i, i + 1])
    return qc


def ideal(qc):
    sv = Statevector(qc)
    return np.array([float(sv.expectation_value(SparsePauliOp("I" * (N - 1 - i) + "Z" + "I" * i)).real)
                     for i in range(N)])


def sampler(backend):
    try:
        from qiskit_ibm_runtime.executor_sampler import Sampler
    except ImportError:
        from qiskit_ibm_runtime import SamplerV2 as Sampler
    return Sampler(mode=backend)


def backend(name):
    return getattr(fp, name)()


def qgz_errors(counts, z):
    # mean |qg_Z - ideal| raw and filtered (weight 1), and the kept fraction
    c = IO.canonical_counts(counts, sdk="qiskit")
    raw = np.mean([abs(q - zi) for (q, _), zi in zip(IO.qg_z_profile(c), z)])
    filt = np.mean([abs(q - zi) for (q, _), zi in zip(IO.filtered_qg_z_profile(c, 1), z)])
    return raw, filt, IO.weight_filter(c, 1)[1]


SEEDS = range(5)          # published: 10 circuits
CIRCS = [circuit(s) for s in SEEDS]
IDEALS = [ideal(qc) for qc in CIRCS]


def transpiled_set(b):
    out = []
    for qc in CIRCS:
        qm = qc.copy()
        qm.measure_all()
        out.append(transpile(qm, b, optimization_level=1, seed_transpiler=137))
    return out

## 3. Coherent conditional-phase errors (§145)

A coherent RZZ(ε) after every **native** two-qubit gate (ECR on Brisbane, CZ on Torino), on top of the device noise. The pre-registered prediction was that the filter cannot see it (it conserves the weight). It failed: inside the native decomposition of the logical XX+YY gate the qubits are not in the computational frame, so part of the error leaves the sector (the kept fraction falls) and is filtered. Published (8000 shots, 10 circuits), raw / filtered at ε = 0 → 0.10: Brisbane 0.103 / 0.024 → 0.107 / 0.025; Torino 0.101 / 0.018 → 0.124 / 0.030.

In [3]:
from qiskit.circuit.library import RZZGate
from qiskit_aer.noise import coherent_unitary_error


def zz_noise_model(b, eps):
    nm = NoiseModel.from_backend(b)
    if eps == 0:
        return nm
    err = coherent_unitary_error(RZZGate(eps).to_matrix())
    gate = next(g for g in ("ecr", "cz", "cx") if g in b.target.operation_names)
    for qs in b.target[gate]:
        if qs is not None and len(qs) == 2:
            nm.add_quantum_error(err, gate, list(qs), warnings=False)
    return nm


print(f"{'backend':14s} {'eps':>5s}   raw    filtered  kept")
for name in BACKENDS:
    b = backend(name)
    ts = transpiled_set(b)
    for eps in (0.0, 0.10):
        res = AerSimulator(noise_model=zz_noise_model(b, eps)).run(ts, shots=4000, seed_simulator=1450).result()
        r, f, k = np.mean([qgz_errors(res.get_counts(i), z) for i, z in enumerate(IDEALS)], axis=0)
        print(f"{name:14s} {eps:5.2f}  {r:.3f}    {f:.3f}   {k:.3f}")

backend          eps   raw    filtered  kept


FakeBrisbane    0.00  0.099    0.019   0.723


FakeBrisbane    0.10  0.103    0.022   0.714


FakeTorino      0.00  0.096    0.023   0.689


FakeTorino      0.10  0.130    0.037   0.661


## 4. Always-on ZZ crosstalk (§150)

Coupled transmons pick up exp(−iθ ZZ/2) all the time, idle or not, θ = 2π ν t. `qang.ibm.add_static_zz` inserts it after every layer of the transpiled circuit on every coupling-map edge between used qubits, with the calibrated layer durations. Between logical gates ZZ is diagonal and invisible to the filter, but most of the time is spent inside the native decompositions, so it is only **partly** invisible. Published at ν = 0 → 15 kHz (raw / filtered, kept): Brisbane 0.102 / 0.023, 0.729 → 0.115 / 0.031, 0.710; Torino (circuits 2.1 µs instead of 9–10 µs) 0.101 / 0.018 → 0.105 / 0.021, almost unchanged.

In [4]:
print(f"{'backend':14s} {'nu (kHz)':>8s}   raw    filtered  kept")
for name in BACKENDS:
    b = backend(name)
    ts = transpiled_set(b)
    sim = AerSimulator(noise_model=NoiseModel.from_backend(b))
    for nu in (0.0, 15e3):
        cs = [ibm.add_static_zz(t, b, nu) for t in ts]
        res = sim.run(cs, shots=4000, seed_simulator=1500).result()
        r, f, k = np.mean([qgz_errors(res.get_counts(i), z) for i, z in enumerate(IDEALS)], axis=0)
        print(f"{name:14s} {nu / 1e3:8.0f}  {r:.3f}    {f:.3f}   {k:.3f}")

backend        nu (kHz)   raw    filtered  kept


FakeBrisbane          0  0.100    0.018   0.728


FakeBrisbane         15  0.115    0.029   0.706


FakeTorino            0  0.098    0.018   0.690


FakeTorino           15  0.104    0.020   0.687


## 5. A PCA spectrum in the weight-1 sector (§146)

In the unary encoding, a mixture of loaded data vectors (weights |x_j|²) is exactly ρ = C / tr C. The RBS circuit of the PCA rotation (`ibm.unary_orthogonal`) puts eigenvalue λ_k on qubit k; every gate conserves the weight, so the filter applies. (This is not the quantum-PCA speed-up: it tests how well a device reads a known spectrum.) Published over 21 devices: the raw estimate is biased upwards (median L1 0.27); filter + readout brings it to 0.048 and gets the order of all five eigenvalues right on 18 of 21 (raw: 13).

In [5]:
D, M, SEED = 5, 16, 146
rng = np.random.default_rng(SEED)
Q, _ = np.linalg.qr(rng.normal(size=(D, D)))
X = rng.normal(size=(M, D)) * np.array((2.0, 1.4, 1.0, 0.6, 0.35)) @ Q.T
X = X - X.mean(axis=0)
O, lam = ibm.pca_rotation(X, center=False)
exact = lam / lam.sum()
w = (X ** 2).sum(axis=1)
w = w / w.sum()
qcs = []
for x in X:
    qc = ibm.unary_loader(x)
    qc.compose(ibm.unary_orthogonal(O), inplace=True)
    qc.measure_all()
    qcs.append(qc)
print("exact spectrum", np.round(exact, 3))
for name in BACKENDS:
    b = backend(name)
    ts = [transpile(qc, b, optimization_level=1, seed_transpiler=137) for qc in qcs]
    res = sampler(b).run(ts, shots=2000).result()
    outs = [ibm.ibm_qg(r, t, b, weight=1) for r, t in zip(res, ts)]
    print(name)
    for k in ("raw", FR):
        est = ibm.unary_spectrum(outs, w, k)[0]
        order = "order ok" if np.all(np.diff(est) < 0) else "order wrong"
        print(f"  {k:17s} {np.round(est, 3)}  L1 {np.abs(est - exact).sum():.3f}  {order}")

exact spectrum [0.64  0.203 0.122 0.023 0.011]


FakeBrisbane
  raw               [0.565 0.272 0.215 0.148 0.172]  L1 0.523  order wrong
  filtered_readout  [0.579 0.205 0.137 0.037 0.042]  L1 0.122  order wrong


FakeTorino
  raw               [0.586 0.23  0.182 0.064 0.097]  L1 0.268  order wrong
  filtered_readout  [0.627 0.202 0.127 0.028 0.016]  L1 0.029  order ok


## 6. Trotter simulation and a quantum walk on the XY chain (§147)

The XY chain with fields conserves the excitation number, and so does every gate of its Trotter circuit (`ibm.xy_trotter`), so the filter applies.

**Part A** (n = 5, T = 3): more Trotter steps r lower the Trotter error but add noise, so the error has a minimum at r*. Published: with filter + readout the best error is 1.6–2.8× lower and r* moves deeper (median 12 against 8).

**Part B** (n = 7, one walker in the middle): a quantum walk spreads ballistically, σ² ∝ t² (exponent 2); a classical random walk diffusively (exponent 1). Published: raw data look classical (median exponent 0.87), filtered data ballistic (1.84; noiseless Trotter 1.94).

In [6]:
NA, T, RS = 5, 3.0, (1, 2, 4, 8, 12)
NB, MID, TIMES, DT = 7, 3, (0.4, 0.8, 1.2, 1.6), 0.2
h = np.random.default_rng(147).uniform(-1, 1, NA)
A = [ibm.xy_trotter(NA, 1.0, h, T, r, initial=(0,)) for r in RS]
B = [ibm.xy_trotter(NB, 1.0, None, t, int(round(t / DT)), initial=(MID,)) for t in TIMES]
exactA = ibm.xy_exact_occupations(NA, 1.0, h, T, 0)


def sigma2(p):
    p = np.clip(np.asarray(p, float), 0, None)
    p = p / p.sum()
    return float(p @ (np.arange(len(p)) - MID) ** 2)


def exponent(s2):
    return float(np.polyfit(np.log(TIMES), np.log(np.maximum(s2, 1e-6)), 1)[0])


s2_ideal = [sigma2([abs(a) ** 2 for a in Statevector(qc.remove_final_measurements(inplace=False)).data[[1 << k for k in range(NB)]]])
            for qc in B]
print("noiseless Trotter: sigma^2 exponent", round(exponent(s2_ideal), 2))
for name in BACKENDS:
    b = backend(name)
    ts = [transpile(qc, b, optimization_level=1, seed_transpiler=137) for qc in A + B]
    res = sampler(b).run(ts, shots=2000).result()
    outs = [ibm.ibm_qg(r, t, b, weight=1) for r, t in zip(res, ts)]
    oa, ob = outs[:len(A)], outs[len(A):]
    print(name)
    for k in ("raw", FR):
        err = [np.abs(ibm.occupations(o, k) - exactA).mean() for o in oa]
        s2 = [sigma2(ibm.occupations(o, k)) for o in ob]
        print(f"  {k:17s} A: errors {np.round(err, 3)}  best {min(err):.3f} at r* = {RS[int(np.argmin(err))]}"
              f"   B: exponent {exponent(s2):.2f}")

noiseless Trotter: sigma^2 exponent 1.94


FakeBrisbane
  raw               A: errors [0.33  0.316 0.195 0.213 0.248]  best 0.195 at r* = 4   B: exponent 0.61
  filtered_readout  A: errors [0.381 0.352 0.136 0.097 0.116]  best 0.097 at r* = 8   B: exponent 1.79


FakeTorino
  raw               A: errors [0.31  0.343 0.192 0.154 0.165]  best 0.154 at r* = 8   B: exponent 0.62
  filtered_readout  A: errors [0.378 0.358 0.144 0.059 0.054]  best 0.054 at r* = 12   B: exponent 1.91


## 7. Interacting excitations: the XXZ chain with 1, 2 and 3 excitations (§148)

The XXZ chain adds V Z_k Z_{k+1}/4 to the XY chain: the excitations interact, and every term still conserves the weight (`ibm.xxz_trotter`). Sectors with more excitations keep more shots, but more of the noise lands *inside* them, where the filter cannot see it. Published (21 devices): the median gain of the filter (best raw / best filtered + readout error) falls from 5.6× with one excitation to 2.6× with two and three. Here: first-order Trotter only, 4 step counts, 2 backends, so the trend is not guaranteed on each backend (on the FakeTorino run saved below the gain stays near 3× for all three sectors; the published fall is a median over 21 devices).

In [7]:
NX, J, V, TX = 6, 1.0, 2.0, 2.0
hx = np.random.default_rng(148).uniform(-0.5, 0.5, NX)
RX = (1, 2, 4, 8)
for name in BACKENDS:
    b = backend(name)
    print(name)
    for wt in (1, 2, 3):
        init = tuple(range(wt))
        ex = ibm.xxz_exact_occupations(NX, J, V, hx, TX, init)
        qcs = [ibm.xxz_trotter(NX, J, V, hx, TX, r, init, 1) for r in RX]
        ts = [transpile(qc, b, optimization_level=1, seed_transpiler=137) for qc in qcs]
        res = sampler(b).run(ts, shots=2000).result()
        outs = [ibm.ibm_qg(r, t, b, weight=wt) for r, t in zip(res, ts)]
        best = {k: min(np.abs(ibm.occupations(o, k) - ex).mean() for o in outs) for k in ("raw", FR)}
        kept = outs[-1]["kept_raw"]
        print(f"  {wt} excitation(s): best raw {best['raw']:.3f}, best filtered+readout {best[FR]:.3f}, "
              f"gain {best['raw'] / best[FR]:.1f}x, kept at r = 8: {kept:.2f}")

FakeBrisbane


  1 excitation(s): best raw 0.132, best filtered+readout 0.029, gain 4.5x, kept at r = 8: 0.23


  2 excitation(s): best raw 0.118, best filtered+readout 0.030, gain 4.0x, kept at r = 8: 0.29


  3 excitation(s): best raw 0.086, best filtered+readout 0.069, gain 1.2x, kept at r = 8: 0.37
FakeTorino


  1 excitation(s): best raw 0.081, best filtered+readout 0.026, gain 3.1x, kept at r = 8: 0.46


  2 excitation(s): best raw 0.065, best filtered+readout 0.023, gain 2.8x, kept at r = 8: 0.51


  3 excitation(s): best raw 0.061, best filtered+readout 0.019, gain 3.1x, kept at r = 8: 0.50


## 8. Swap test vs inversion test (§149)

Two ways to estimate the fidelity F = |⟨x|y⟩|² of two unary states. The **inversion test** (load x, undo the loader of y, P(e₀) = F) uses 5 qubits and only weight-preserving gates: the filter applies fully. The **swap test** uses 11 qubits; its controlled SWAPs conserve the weight of the data registers, but the answer is on the ancilla, outside the symmetry. Published (6 devices): the filter cuts the inversion-test error 3.8–11.2×, the swap-test error only 1.4–1.8×; a prototype classifier (iris versicolor / virginica) keeps its score either way, because comparing two fidelities cancels the common shrinkage, but for identical states the filtered inversion test reads 0.92–1.00 and the raw swap test 0.20–0.65. Here: 10 test flowers and 5 identical pairs.

In [8]:
from sklearn.datasets import load_iris

Xi, yi = load_iris(return_X_y=True)
m = yi > 0
Xi, yi = Xi[m], yi[m] - 1
rng = np.random.default_rng(149)
tr, te = [], []
for c in (0, 1):
    idx = rng.permutation(np.where(yi == c)[0])
    tr += list(idx[:15])
    te += list(idx[15:20])
mu, sd = Xi[tr].mean(0), Xi[tr].std(0)
emb = lambda v: (lambda e: e / np.linalg.norm(e))(np.append((v - mu) / sd, 1.0))
E = np.array([emb(v) for v in Xi])
P = [E[[i for i in tr if yi[i] == c]].mean(0) for c in (0, 1)]
P = [p / np.linalg.norm(p) for p in P]
pairs = [(E[i], P[c]) for i in te for c in (0, 1)] + [(E[i], E[i]) for i in te[:5]]
exactF = np.array([(x @ y) ** 2 for x, y in pairs])
labels = yi[te]
nte = 2 * len(te)


def accuracy(F):
    F = np.asarray(F[:nte]).reshape(-1, 2)
    return int(((F[:, 1] > F[:, 0]).astype(int) == labels).sum())


print(f"noiseless accuracy {accuracy(exactF)}/{len(te)}")
for name in BACKENDS:
    b = backend(name)
    print(name)
    for test, make, read in (("inversion", ibm.unary_overlap_circuit, ibm.overlap_fidelity),
                             ("swap", ibm.swap_test_circuit, ibm.swap_test_fidelity)):
        ts = [transpile(make(x, y), b, optimization_level=1, seed_transpiler=137) for x, y in pairs]
        res = sampler(b).run(ts, shots=2000).result()
        fids = [read(r, t, b) for r, t in zip(res, ts)]
        for k in ("raw", FR):
            est = np.array([f[k] for f in fids], float)
            print(f"  {test:9s} {k:17s} error {np.nanmean(np.abs(est - exactF)):.3f}  "
                  f"accuracy {accuracy(est)}/{len(te)}  identical pairs read {np.nanmean(est[nte:]):.2f}")
print(f"total time {time.time() - T0:.0f} s")

noiseless accuracy 9/10
FakeBrisbane


  inversion raw               error 0.158  accuracy 9/10  identical pairs read 0.66
  inversion filtered_readout  error 0.047  accuracy 9/10  identical pairs read 0.91


  swap      raw               error 0.295  accuracy 9/10  identical pairs read 0.39
  swap      filtered_readout  error 0.178  accuracy 9/10  identical pairs read 0.61
FakeTorino


  inversion raw               error 0.145  accuracy 9/10  identical pairs read 0.72
  inversion filtered_readout  error 0.011  accuracy 9/10  identical pairs read 1.00


  swap      raw               error 0.260  accuracy 9/10  identical pairs read 0.46
  swap      filtered_readout  error 0.145  accuracy 9/10  identical pairs read 0.71
total time 823 s


## Summary

* **Coherent errors (§145, §150)** are filtered in proportion to the time the qubits spend outside the computational frame: an error on the native gate, or always-on ZZ during the native decompositions, partly leaves the sector; ZZ between logical gates does not. Shorter circuits (Torino) suffer least from always-on ZZ.
* **Reading a spectrum (§146)**: the raw estimate is biased upwards by shots that leave the sector; filter + readout removes most of the bias.
* **Dynamics (§147, §148)**: the filter lets a Trotter simulation use deeper circuits and keeps a quantum walk ballistic; its gain shrinks as the sector holds more excitations, because more noise stays inside it.
* **Fidelities (§149)**: put the answer inside the symmetry. The inversion test benefits fully from the filter; the swap test, whose answer is on an ancilla, hardly does.

Full details, pre-registered predictions and all backends: `RESEARCH_NOTES.md` §145–§150 and the scripts in `examples/`.